# Teste externo

Compara os modelos salvos em `modelos_pt/` com imagens de **outra fonte**, que não vieram dos datasets do treino. É o teste que mostra se o modelo funciona fora do dataset.

## Como usar

1. Coloque as imagens em `data/teste_externo/<cancer>/<classe>/`, com as pastas de classe iguais às do treino. Exemplo: `data/teste_externo/mama/benign/` e `data/teste_externo/mama/malignant/`.
2. Use de preferência de 20 a 30 imagens por classe, com rótulo confiável e uso permitido. Anote a fonte de cada conjunto.
3. Ajuste `cancer` na célula de configuração e rode tudo. Não precisa de GPU.

Todos os arquivos `modelos_pt/<cancer>_*.pt` entram na comparação. Não use estas imagens para escolher ajustes finos de treino repetidamente, senão elas deixam de ser um teste independente.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from torch import Tensor
from torch.nn import AdaptiveAvgPool2d, Conv2d, Dropout, Flatten, Linear, MaxPool2d, Module, ReLU, Sequential

In [ ]:
# Prefixo dos arquivos em modelos_pt/ e nome da pasta em data/teste_externo/
cancer = "mama"

# Docker
# modelos_dir = Path("/app/modelos_pt")
# teste_dir = Path("/app/data/teste_externo") / cancer
# VS Code (PC)
modelos_dir = Path("../modelos_pt")
teste_dir = Path("../data/teste_externo") / cancer

extensoes = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

In [ ]:
class CustomModel(Module):
    # Mesma arquitetura dos notebooks; o número de canais vem do modelo salvo (3 na cor, 1 no cinza)
    def __init__(self, canais: int, output_size: int) -> None:
        super().__init__()

        self.net = Sequential(
            Conv2d(canais, 16, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
            Conv2d(16, 32, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
            Conv2d(32, 64, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
            AdaptiveAvgPool2d(1),
            Flatten(),
            Dropout(p=0.25),
            Linear(64, output_size),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


def carregar_modelo(caminho: Path):
    dados = torch.load(caminho, map_location="cpu", weights_only=True)
    canais = 3 if dados["modo"] == "cor" else 1
    modelo = CustomModel(canais, len(dados["class_names"]))
    modelo.load_state_dict(dados["state_dict"])
    modelo.eval()
    return modelo, dados


def preparar_imagem(caminho: Path, modo: str, size: int, escala: float) -> torch.Tensor:
    # Mesmo pré-processamento do load_data dos notebooks e da API do PixoraMed
    flag = cv2.IMREAD_COLOR if modo == "cor" else cv2.IMREAD_GRAYSCALE
    # imdecode em vez de imread: funciona com acentos no caminho do arquivo no Windows
    imagem = cv2.imdecode(np.fromfile(str(caminho), dtype=np.uint8), flag)
    if imagem is None:
        raise ValueError(f"Não foi possível ler: {caminho}")
    if modo == "cor":
        imagem = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)
    imagem = cv2.resize(imagem, (size, size))
    imagem = imagem.transpose(2, 0, 1) if modo == "cor" else imagem[np.newaxis]
    return torch.from_numpy(imagem.copy()).float().unsqueeze(0) / escala

In [ ]:
def listar_imagens(class_names):
    imagens = []
    for indice, classe in enumerate(class_names):
        pasta = teste_dir / classe
        if not pasta.is_dir():
            raise FileNotFoundError(f"Crie a pasta {pasta} com imagens da classe {classe}")
        arquivos = sorted(p for p in pasta.iterdir() if p.suffix.lower() in extensoes)
        if not arquivos:
            raise FileNotFoundError(f"Sem imagens em {pasta}")
        imagens += [(arquivo, indice) for arquivo in arquivos]
    return imagens


caminhos_modelos = sorted(modelos_dir.glob(f"{cancer}_*.pt"))
if not caminhos_modelos:
    raise FileNotFoundError(f"Nenhum modelo {cancer}_*.pt em {modelos_dir.resolve()}")

linhas = []
matrizes = {}
erros = []
for caminho in caminhos_modelos:
    modelo, dados = carregar_modelo(caminho)
    class_names = list(dados["class_names"])
    imagens = listar_imagens(class_names)
    y_true, y_pred, confiancas = [], [], []
    with torch.no_grad():
        for arquivo, real in imagens:
            entrada = preparar_imagem(arquivo, dados["modo"], dados["size"], dados["escala"])
            probabilidades = torch.softmax(modelo(entrada), dim=1)[0]
            previsto = int(probabilidades.argmax())
            y_true.append(real)
            y_pred.append(previsto)
            confiancas.append(float(probabilidades[previsto]))
            if previsto != real:
                erros.append({"modelo": caminho.stem, "imagem": f"{class_names[real]}/{arquivo.name}", "previsto": class_names[previsto], "confiança_%": round(confiancas[-1] * 100, 1)})

    matriz = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    matrizes[caminho.stem] = pd.DataFrame(matriz, index=[f"real {c}" for c in class_names], columns=[f"prev. {c}" for c in class_names])
    acerto_classe = matriz.diagonal() / matriz.sum(axis=1)
    linhas.append({
        "modelo": caminho.stem,
        "modo": dados["modo"],
        "normalizado": "Sim" if dados["escala"] == 255 else "Não",
        "imagens": len(imagens),
        "acc_%": round(accuracy_score(y_true, y_pred) * 100, 1),
        "f1_macro_%": round(f1_score(y_true, y_pred, average="macro") * 100, 1),
        "pior_classe_%": round(float(acerto_classe.min()) * 100, 1),
        "confiança_média_%": round(float(np.mean(confiancas)) * 100, 1),
        "acc_5_folds_%": dados.get("metricas", {}).get("acc_media"),
    })

resultado = pd.DataFrame(linhas).sort_values(["f1_macro_%", "pior_classe_%"], ascending=False)
print(resultado.to_string(index=False))

In [ ]:
for nome, matriz in matrizes.items():
    print(f"\n{nome}\n{matriz.to_string()}")

print(f"\nErros: {len(erros)}")
if erros:
    print(pd.DataFrame(erros).to_string(index=False))

saida = teste_dir / "resultado_teste_externo.csv"
resultado.to_csv(saida, sep=";", decimal=",", index=False, encoding="utf-8-sig")
print(f"\nResultado salvo em: {saida.resolve()}")